In [1]:
# 1. import libraries
import pandas as pd                
import nltk                         
import string                         
import matplotlib.pyplot as plt      
import seaborn as sns                 
import numpy as np                  
from nltk.corpus import stopwords
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

nltk.download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
# 2. LOAD DATA
data = fetch_20newsgroups(subset='all')

df = pd.DataFrame({
    'text':   data.data,    
    'target': data.target   
})

print(f'Total articles : {len(df)}')
print(f'Total topics : {len(data.target_names)}')
print(f'Topics : {data.target_names}')

Total articles : 18846
Total topics : 20
Topics : ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [3]:
# 3. CLEAN TEXT
stop_words = set(stopwords.words('english'))  

def clean_text(text):
    text = text.lower()                                                 
    text = "".join([c for c in text if c not in string.punctuation])     
    words = text.split()                                               
    words = [w for w in words if w not in stop_words]                  
    return " ".join(words)

df['clean_text'] = df['text'].apply(clean_text)

print('before:\n', df['text'].iloc[0][:200])
print('=======================')
print('after:\n', df['clean_text'].iloc[0][:200])

before:
 From: Mamatha Devineni Ratnam <mr47+@andrew.cmu.edu>
Subject: Pens fans reactions
Organization: Post Office, Carnegie Mellon, Pittsburgh, PA
Lines: 12
NNTP-Posting-Host: po4.andrew.cmu.edu



I am sur
after:
 mamatha devineni ratnam mr47andrewcmuedu subject pens fans reactions organization post office carnegie mellon pittsburgh pa lines 12 nntppostinghost po4andrewcmuedu sure bashers pens fans pretty confu


In [4]:
df

,text,target,clean_text
0,From: Mamatha Devineni Ratnam <mr47+@andrew.cm...,10,mamatha devineni ratnam mr47andrewcmuedu subje...
1,From: mblawson@midway.ecn.uoknor.edu (Matthew ...,3,mblawsonmidwayecnuoknoredu matthew b lawson su...
2,From: hilmi-er@dsv.su.se (Hilmi Eren)\nSubject...,17,hilmierdsvsuse hilmi eren subject armenia says...
3,From: guyd@austin.ibm.com (Guy Dawson)\nSubjec...,3,guydaustinibmcom guy dawson subject ide vs scs...
4,From: Alexander Samuel McDiarmid <am2o+@andrew...,4,alexander samuel mcdiarmid am2oandrewcmuedu su...
...,...,...,...
18841,From: jim.zisfein@factory.com (Jim Zisfein) \n...,13,jimzisfeinfactorycom jim zisfein subject migra...
18842,From: rdell@cbnewsf.cb.att.com (richard.b.dell...,12,rdellcbnewsfcbattcom richardbdell subject ques...
18843,From: westes@netcom.com (Will Estes)\nSubject:...,3,westesnetcomcom estes subject mounting cpu coo...
18844,From: steve@hcrlgw (Steven Collins)\nSubject: ...,1,stevehcrlgw steven collins subject sphere 4 po...


In [5]:
# 4. SPLIT DATa
X = df['clean_text']   
y = df['target']       
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,random_state=42     
)
X_test_raw = X_test.copy()  

In [6]:
# 5. TF-IDF
vectorizer = TfidfVectorizer(max_features=3000) 
X_train = vectorizer.fit_transform(X_train)  
X_test  = vectorizer.transform(X_test)       

In [7]:
# 6. MODEL_Naive Bayes
model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [8]:
import random

def predict_random_article():
    
    idx = random.randint(0, len(X_test_raw) - 1)

    article_text  = X_test_raw.iloc[idx]
    true_label_id = y_test.iloc[idx]
    true_label    = data.target_names[true_label_id]
    
    article_vec = vectorizer.transform([article_text])  

    pred_label_id = model.predict(article_vec)[0]
    pred_label    = data.target_names[pred_label_id]


    print("-" * 60)
    print(f"Article #{idx}")
    print("-" * 60)
    print(f"True Category : {true_label}")
    print(f"Predicted     : {pred_label}")
    print(f"Correct?      : {'YES' if true_label == pred_label else 'NO'}")

    print("\nText:\n")
    print(article_text[:500])  



p
redict_random_article()

NameError: name 'p' is not defined

In [ ]:
# 7. EVALUATION  
y_pred = model.predict(X_test)  
#---------------
accuracy  = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average='weighted'
)

print('Model Evaluation Results')
print('-' * 50)
print(f'Accuracy  :    {accuracy:.4f}   |  {accuracy*100:.2f}%')
print(f'Precision :    {precision:.4f}   |  {precision*100:.2f}%')
print(f'Recall    :    {recall:.4f}   |  {recall*100:.2f}%')
print(f'F1-Score  :    {f1:.4f}   |  {f1*100:.2f}%')
print('-' * 50)

In [ ]:
#Overall Metrics 
metrics = ['F1-Score','Precision', 'Recall', 'Accuracy']
values  = [f1, precision, recall,accuracy]
colors  = ['#4f86c6', '#6dbf67', '#f4a261', '#e76f51']

plt.figure(figsize=(8, 5))  
bars = plt.barh(
    metrics,
    values,
    color=colors,
    height=0.7,          
    edgecolor='white',
    linewidth=1.5
)

for i, v in enumerate(values):
    plt.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=11)

plt.title('Model Evaluation Metrics')
plt.xlabel('Score')
plt.xlim(0, 1.1)
plt.show()

In [ ]:
#F1 score per class


# F1 
cat_names = data.target_names
report_dict = classification_report(
    y_test, y_pred,
target_names= cat_names,
    output_dict=True
)



f1_per_class = [report_dict[c]['f1-score'] for c in cat_names]


sorted_idx = np.argsort(f1_per_class)
sorted_names = [cat_names[i] for i in sorted_idx]
sorted_f1 = [f1_per_class[i] for i in sorted_idx]
#-----------------------
plt.figure(figsize=(10, 6))
bar_colors = [
    '#e63946' if v < 0.75 else
    '#f4a261' if v < 0.85 else
    '#52b788'
    for v in sorted_f1
]

bars = plt.barh(sorted_names, sorted_f1, color=bar_colors)

for i, v in enumerate(sorted_f1):
    plt.text(v + 0.01, i, f'{v:.2f}', va='center')

plt.xlim(0, 1.1)
plt.title('F1-Score per Category')
plt.xlabel('F1 Score')

plt.tight_layout()
plt.show()

In [ ]:
#confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=False,          
    cmap='Blues',
    xticklabels=cat_names,
    yticklabels=cat_names,
    linewidths=0.5,       
    linecolor='white'
)

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')

plt.tight_layout()
plt.show()